# Exercise: Contextual String Embeddings for Sequence Labeling (Akbik et al., 2018)

In the seminar we covered Flair contextual string embeddings and the BiLSTM-CRF sequence labeler.
This short exercise focuses on understanding the architecture and implementing the CRF layer.


In [2]:
def TODO(todo: str = "Fill the blank"):
    raise ValueError(todo)

## Exercise Part 1: Akbik et al. recap and CRF


### Task 1.1

Read the paper of Akbik et al. Then answer briefly:

1. What is the key difference between static word embeddings (e.g. GloVe) and contextual string embeddings?
2. Why is a BiLSTM used also as sequence tagger on top of the contextual word embeddings? How does its performance compare to a standard feedforward network?
3. What problem does a Conditional Random Field (CRF; eq. 10 & 11) solve that independent token classification does not?


### Answers

**1. Static vs. contextual embeddings**

Static embeddings like GloVe assign a single fixed vector to each word *type*, regardless of the sentence it appears in. The word "bank" always receives the same vector, even if it refers to a riverbank in one sentence and a financial institution in another.

Contextual string embeddings are derived from the hidden states of a character-level language model that processes the entire sentence. Because the hidden state encodes everything seen so far (forward LM) or everything seen from the right (backward LM), the resulting embedding for a word depends on its surrounding context. The same word string can therefore receive different embeddings in different sentences.

Additionally, since the model operates at the character level without an explicit word vocabulary, it naturally handles rare and misspelled words as well as morphological subword structure (prefixes, suffixes, etc.).

**2. BiLSTM on top**

Even though the contextual embeddings already encode rich context, a BiLSTM on top can still learn useful sequential patterns *across* positions that are relevant to the labeling task specifically — for example, that certain syntactic constructions strongly predict a particular tag sequence.

The paper's ablation study (Table 3) shows that removing the BiLSTM and replacing it with a direct linear map (feedforward) drops F-score by only ~3 pp for the proposed contextual embeddings, versus ~20 pp for classic GloVe embeddings. This means the BiLSTM matters far more when the underlying embeddings are less informative, but still provides a consistent improvement on top of contextual embeddings.

**3. What the CRF adds**

Independent token classification (e.g. softmax at each position separately) treats each label decision in isolation. It cannot enforce structural constraints over the label sequence. For example, in BIO tagging it is invalid to have an `I-PER` tag immediately follow a `B-LOC` tag, but a per-token softmax has no mechanism to discourage this.

A CRF (eqs. 10–11) models the *joint* probability over the entire output sequence. It learns transition scores between pairs of tags, allowing the model to reward or penalize certain tag transitions globally. Decoding then finds the globally optimal sequence (via Viterbi) rather than greedily picking the best label at each step independently.


--- 

### Task 1.2 
Assume a BiLSTM produced the following emission scores.


In [9]:
TOKENS = ["George", "Lewis", "lives", "in", "Berlin"]

EMISSIONS = [
    {"B-PER": 5, "I-PER": 4, "O": 1},   # George
    {"B-PER": 2, "I-PER": 1, "O": 0},   # Lewis
    {"B-LOC": 1, "I-LOC": 1, "O": 5},   # lives
    {"B-LOC": 2, "I-LOC": 1, "O": 6},   # in
    {"B-LOC": 6, "I-LOC": 1, "O": 2},   # Berlin
]

### Question

If we classify each token independently, which labels would be chosen?

In [10]:
# Solution: greedy independent classification
greedy_labels = [max(emission, key=emission.get) for emission in EMISSIONS]

for token, label in zip(TOKENS, greedy_labels):
    print(f"{token:10s} -> {label}")

George     -> B-PER
Lewis      -> B-PER
lives      -> O
in         -> O
Berlin     -> B-LOC


**Expected output:**
```
George     -> B-PER   (score 5, highest)
Lewis      -> B-PER   (score 2, highest)
lives      -> O       (score 5, highest)
in         -> O       (score 6, highest)
Berlin     -> B-LOC   (score 6, highest)
```

### Task 1.3

The CRF additionally learns transition scores. Assume the following scores were learned by the model:


In [11]:
TRANSITIONS = {
    ("I-PER", "B-PER"): -20,
    ("B-PER", "B-PER"): -3,
    ("B-PER", "I-PER"): 1,
    ("B-PER", "O"):     0,
    ("O",     "O"):     0,
    ("O",     "B-LOC"): 2,
    ("B-LOC", "O"):    -2,
    ("B-LOC", "B-LOC"): -1,
}

The score of a sequence is:
$$ \text{score} =
\text{sum of emission scores}
+
\text{sum of transition scores}$$

### Task

Complete the implementation. Then compare the candidate sequences.

In [12]:
def sequence_score(tags, emissions, transitions):
    """Compute the CRF score for a tag sequence.
    
    Score = sum of emission scores + sum of transition scores.
    Missing transitions are treated as 0 (neutral).
    """
    score = emissions[0][tags[0]]  # emission score for the first token

    for i in range(1, len(tags)):
        score += emissions[i][tags[i]]                         # emission score at position i
        score += transitions.get((tags[i - 1], tags[i]), 0)   # transition from tags[i-1] to tags[i]

    return score

In [13]:
candidate_1 = ["B-PER", "O",     "O", "O", "B-LOC"]  # George=PER, Lewis=O, ...  (greedy-style)
candidate_2 = ["B-PER", "B-PER", "O", "O", "B-LOC"]  # Both names tagged as B-PER
candidate_3 = ["B-PER", "I-PER", "O", "O", "B-LOC"]  # Both names form a PER span

for candidate in [candidate_1, candidate_2, candidate_3]:
    print(f"Candidate: {candidate}")
    print(f"  Score:   {sequence_score(candidate, EMISSIONS, TRANSITIONS)}")
    print()

Candidate: ['B-PER', 'O', 'O', 'O', 'B-LOC']
  Score:   24

Candidate: ['B-PER', 'B-PER', 'O', 'O', 'B-LOC']
  Score:   23

Candidate: ['B-PER', 'I-PER', 'O', 'O', 'B-LOC']
  Score:   26



### Questions

**Which sequence receives the highest score?**  
Candidate 1 (`B-PER, I-PER, O, O, B-LOC`) scores highest at **26**.

**Why is candidate 2 penalized?**  
Candidate 2 uses two consecutive `B-PER` tags. The transition `(B-PER → B-PER)` carries a score of **−3**, which the model learned to penalize. In BIO tagging, a `B-` tag signals the *beginning* of a new entity, so seeing another `B-PER` immediately after suggests a new entity started — an unusual and typically invalid pattern for a two-word person name. The CRF can encode this structural knowledge through the learned transition matrix.

# Part 2: CRF in Pytorch

Given emission scores $\mathbf{e} \in \mathbb{R}^{T \times K}$ from the BiLSTM and a transition matrix $\mathbf{A} \in \mathbb{R}^{K \times K}$, the score of a tag sequence $\mathbf{y} = (y_0, \dots, y_{T-1})$ is

$$s(\mathbf{y}) = \sum_{t=0}^{T-1} e_t[y_t] \;+\; \sum_{t=1}^{T-1} A[y_{t-1},\, y_t]$$

#### Task: 

Finish below pytorch implementation of a CRF scorer. Initialize the transition matrix randomly and make sure it can be trained (Hint: Check pytorch documentation for nn.Parameter)

In [14]:
import torch
import torch.nn as nn

# ── CRF (Eq. 10-11) ───────────────────────────────────────────────────────
class CRF(nn.Module):
    def __init__(self, num_tags):
        super().__init__()
        self.num_tags    = num_tags
        self.transitions = nn.Parameter(torch.randn(num_tags, num_tags) * 0.1)

    def score(self, emit_scores, tags):
        score = emit_scores[0, tags[0]]
        for t in range(1, emit_scores.size(0)):
            score += self.transitions[tags[t], tags[t-1]] + emit_scores[t, tags[t]]
        return score